# Lab 4: Leveraging Open Data from Wikipedia for LLM Prompt Engineering

## Overview
This lab demonstrates how to extract structured data from Wikipedia pages and use it to create effective prompts for Large Language Models (LLMs). You'll learn to work with real-world financial data, process it programmatically, and engineer prompts for various AI tasks.

## Learning Objectives
- ✓ Extract financial index components from Wikipedia
- ✓ Retrieve company infobox data programmatically
- ✓ Build structured datasets from semi-structured web data
- ✓ Design effective LLM prompts for different tasks
- ✓ Process and clean text data for AI consumption
- ✓ Create reusable prompt templates and utilities

By Kerrian Le Bars

## Part 1: Data Extraction from Wikipedia

### What is a Financial Index?
A financial index is a composite measure of a subset of companies in a specific market or sector. Examples include:
- **S&P 500**: 500 largest US companies
- **EURO STOXX 50**: 50 largest Eurozone companies
- **DAX**: 40 largest German companies

### Your Task
1. **Identify components**: Extract the list of companies in each index from Wikipedia
2. **Gather company data**: Retrieve detailed information (infoboxes) from each company's Wikipedia page
3. **Build a dataset**: Combine all data into structured format suitable for LLM processing
4. **Engineer prompts**: Create effective prompts that leverage this data for AI tasks

### Data Sources
- **Index components**: Wikipedia articles listing index members
- **Company data**: Wikipedia infoboxes (structured data boxes on company pages)
- **Dump file**: Optional - for advanced analysis of full Wikipedia articles

### Optional: Full Wikipedia Dump
For advanced analysis, you can download the complete Wikipedia dump from:
- **Link**: https://dumps.wikimedia.org/enwiki/
- **File**: `enwiki-latest-pages-articles-multistream-index.txt.bz2`
- **Use case**: Full-text search, article history analysis, or complete data scraping
- **Note**: Very large files (100+ GB) - requires significant storage and processing power

For this lab, we'll focus on extracting specific data via the Wikipedia API, which is more efficient.

In [50]:
# ============================================================================
# IMPORTS & SETUP
# ============================================================================
# These libraries enable us to work with Wikipedia data

import pandas as pd                                   # Data manipulation and analysis
import urllib.request                                 # HTTP requests to Wikipedia
from pathlib import Path                              # Cross-platform file path handling
from typing import Union, Optional, Dict, Any, List   # Type hints for better code clarity
from tqdm import tqdm                                 # Progress bars for long operations
import wptools                                        # Wikipedia parsing (infobox extraction)
from loguru import logger                             # Enhanced logging
import json                                           # Working with JSON data
import numpy as np                                    # Numerical operations
import time                                           # Timing delays (respectful API scraping)
import re                                             # Regular expressions for text processing
from datetime import datetime                         # Date and time handling


## Step 1: Extract Index Components from Wikipedia

### Task: Extract Company Lists
We'll extract the list of companies that make up each financial index directly from Wikipedia.

### Indices We're Covering:
1. **S&P 500** (USA) - 500 largest US companies
2. **EURO STOXX 50** (Europe) - 50 largest Eurozone companies  
3. **CAC 40** (France) - 40 largest French companies
4. **DAX** (Germany) - 40 largest German companies
5. **CSI 300** (China) - 300 largest Chinese companies
6. **S&P Latin America 40** (Latin America) - 40 major LA companies
7. **BSE SENSEX** (India) - 30 largest Indian companies
8. **NASDAQ-100** (USA Tech) - 100 largest non-financial NASDAQ companies

### How It Works:
- Each index has a Wikipedia article with a table listing its components
- We'll use `pd.read_html()` to extract all tables from these pages
- Tables are saved as CSV files for later processing
- This approach is fast, requires no authentication, and respects Wikipedia's terms

In [26]:
# ============================================================================
# FUNCTION 1: Extract Tables from Wikipedia
# ============================================================================
# This function downloads tables from Wikipedia articles and saves them locally

def get_index_components(wiki_url: str, save_dir: Union[str, Path], 
                         opener: urllib.request.OpenerDirector) -> None:
    """
    Extract all HTML tables from a Wikipedia page and save as CSV files.
    
    Parameters:
    -----------
    wiki_url : str
        The Wikipedia page URL to scrape (e.g., list of index components)
    save_dir : Union[str, Path]
        Directory where CSV files will be saved
    opener : urllib.request.OpenerDirector
        Custom URL opener with proper User-Agent headers
        
    Output:
    -------
    Creates CSV files named table_0.csv, table_1.csv, etc. in save_dir
    Each file contains one table from the Wikipedia page
    
    Example:
    --------
    >>> get_index_components(
    ...     "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    ...     "./data/indices/sp500",
    ...     opener
    ... )
    """

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    
    # Fetch the Wikipedia page using the opener
    with opener.open(wiki_url) as response:
        html_content = response.read().decode('utf-8')
    
    # Extract all tables from the HTML using pandas
    tables = pd.read_html(html_content)
    
    # Save each table as a CSV file
    for i, table in enumerate(tables):
        csv_path = save_path / f"table_{i}.csv"
        table.to_csv(csv_path, index=False)
    
    logger.info(f"Extracted {len(tables)} tables from {wiki_url} -> {save_dir}")

In [27]:
# ============================================================================
# SETUP: Configure Wikipedia Index URLs and HTTP Headers
# ============================================================================

# Dictionary mapping index names to their Wikipedia article URLs
# These URLs contain tables with the company components of each index
indices = {
    "sp500": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    "eurostoxx50": "https://en.wikipedia.org/wiki/EURO_STOXX_50",
    "cac40": "https://en.wikipedia.org/wiki/CAC_40",
    "dax": "https://en.wikipedia.org/wiki/DAX",
    "csi300": "https://en.wikipedia.org/wiki/CSI_300_Index",
    "spla40": "https://en.wikipedia.org/wiki/S%26P_Latin_America_40",
    "bsesensex": "https://en.wikipedia.org/wiki/BSE_SENSEX",
    "nasdaq100": "https://en.wikipedia.org/wiki/Nasdaq-100",
}

# IMPORTANT: Configure HTTP headers to identify our bot to Wikipedia
# This is REQUIRED for ethical web scraping - identify yourself!
# Wikipedia may block requests without proper User-Agent headers

opener = urllib.request.build_opener()
opener.addheaders = [
    ("User-Agent", "MyResearchBot/1.0 (contact@example.com)")  # Identify your bot
]
urllib.request.install_opener(opener)

In [28]:
# ============================================================================
# EXECUTION: Download Index Components
# ============================================================================
# Loop through each index and extract its company components from Wikipedia
# This may take a few minutes depending on internet speed

for index_name, wiki_url in tqdm(indices.items(), desc="Downloading indices"):
    save_dir = Path(f"./sample_data/indices/{index_name}")
    get_index_components(wiki_url, save_dir, opener)

  tables = pd.read_html(html_content)
2025-11-28 11:46:13.719 | INFO     | __main__:get_index_components:49 - Extracted 3 tables from https://en.wikipedia.org/wiki/List_of_S%26P_500_companies -> sample_data\indices\sp500
  tables = pd.read_html(html_content)
2025-11-28 11:46:13.943 | INFO     | __main__:get_index_components:49 - Extracted 10 tables from https://en.wikipedia.org/wiki/EURO_STOXX_50 -> sample_data\indices\eurostoxx50
  tables = pd.read_html(html_content)
2025-11-28 11:46:14.306 | INFO     | __main__:get_index_components:49 - Extracted 20 tables from https://en.wikipedia.org/wiki/CAC_40 -> sample_data\indices\cac40
  tables = pd.read_html(html_content)
2025-11-28 11:46:14.575 | INFO     | __main__:get_index_components:49 - Extracted 10 tables from https://en.wikipedia.org/wiki/DAX -> sample_data\indices\dax
  tables = pd.read_html(html_content)
2025-11-28 11:46:14.961 | INFO     | __main__:get_index_components:49 - Extracted 7 tables from https://en.wikipedia.org/wiki/CSI_

## Step 2: Extract Company Infoboxes from Wikipedia

### What are Infoboxes?
Wikipedia infoboxes are structured data boxes that appear on the right side of articles. They contain:
- Company name and alternative names
- Industry classification
- Founded date and location
- Key executives
- Headquarters location
- Number of employees
- Revenue and financial metrics
- Official website URLs
- Stock exchange listings
- And much more...

### Why Infoboxes?
- **Structured data**: Unlike article body text, infoboxes are semi-structured
- **Consistency**: Fields follow a template across similar articles
- **Ease of extraction**: Wikipedia APIs can parse infoboxes directly
- **Rich context**: Perfect for LLM prompts - contains exactly the info LLMs need

### Process
1. Use the `wptools` library to fetch each company's Wikipedia page
2. Extract the infobox (structured data) from the page parse
3. Save as JSON for flexibility and later processing
4. Handle errors gracefully (some companies may not have Wikipedia pages)

In [29]:
# ============================================================================
# FUNCTION 2: Fetch Company Infobox from Wikipedia
# ============================================================================
# This function retrieves the structured "infobox" section of a company's
# Wikipedia page using the wptools library. Infoboxes contain standardized
# company metadata such as industry, headquarters, revenue, founders, etc.

def fetch_company_infobox(company_name: str, 
                          delay: float = 0.5) -> Optional[Dict[str, Any]]:
    """
    Fetch the infobox for a company from Wikipedia using wptools.

    Parameters
    ----------
    company_name : str
        Name of the company (as appears on Wikipedia)
    delay : float
        Pause between API calls (respectful scraping)

    Returns
    -------
    dict or None
        Parsed infobox dictionary, or None if the page is missing.
    """

    try:
        page = wptools.page(company_name)
        page.get_parse()  # fetch & parse the page

        time.sleep(delay)  # Pause to avoid hammering Wikipedia

        if page.data.get("infobox"):
            return page.data["infobox"]
        else:
            print(f"No infobox found for: {company_name}")
            return None

    except Exception as e:
        print(f"Error fetching infobox for {company_name}: {e}")
        return None

In [30]:
# ============================================================================
# EXAMPLE: Extract a Single Company Infobox
# ============================================================================
# This example shows the process for one company (3M) from S&P 500
# In production, we'd loop this for all companies

company = "3M"  # Example from S&P 500
output_dir = Path("./sample_data/infobox_examples")

infobox = fetch_company_infobox(company)

en.wikipedia.org (parse) 3M
en.wikipedia.org (imageinfo) File:3-M Building Maplewood MN1.jpg
3M (en) data
{
  image: <list(1)> {'kind': 'parse-image', 'file': 'File:3-M Build...
  infobox: <dict(24)> name, logo, logo_size, image, image_size, im...
  iwlinks: <list(1)> https://commons.wikimedia.org/wiki/Category:3M
  pageid: 7664801
  parsetree: <str(101035)> <root><template><title>Short descriptio...
  requests: <list(2)> parse, imageinfo
  title: 3M
  wikibase: Q159433
  wikidata_url: https://www.wikidata.org/wiki/Q159433
  wikitext: <str(82081)> {{Short description|American multinationa...
}


In [31]:
# ============================================================================
# FUNCTION 3: Save Company Infobox as JSON
# ============================================================================
# This function stores the extracted infobox for a given company into a JSON
# file. JSON is ideal for structured storage because infobox fields vary
# between companies and may be nested or semi-structured.

def save_infobox_json(company_name: str, 
                      infobox: Dict[str, Any], 
                      output_dir: Path) -> None:
    """
    Save one company's infobox into a JSON file.

    Parameters
    ----------
    company_name : str
        Company name
    infobox : dict
        Parsed infobox data
    output_dir : Path
        Directory where JSON files will be stored
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    file_path = output_dir / f"{company_name.replace(' ', '_')}.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(infobox, f, indent=4, ensure_ascii=False)

    print(f"✔ Saved infobox: {file_path}")

In [32]:
# ============================================================================
# DISPLAY: View the Extracted Infobox
# ============================================================================
# This shows what data we extracted from Wikipedia

if infobox is not None:
    save_infobox_json(company, infobox, output_dir)

print("\n=== Extracted Infobox for 3M ===")
infobox

✔ Saved infobox: sample_data\infobox_examples\3M.json

=== Extracted Infobox for 3M ===


{'name': '3M Company',
 'logo': '3M wordmark.svg',
 'logo_size': '175px',
 'image': '3-M Building Maplewood MN1.jpg',
 'image_size': '250px',
 'image_caption': '3M headquarters in [[Maplewood, Minnesota]]',
 'former_name': 'Minnesota Mining and Manufacturing Company (1902–2002)',
 'type': '[[Public company|Public]]',
 'traded_as': '{{Unbulleted list|New York Stock Exchange|MMM|[[Dow Jones Industrial Average|DJIA]] component|[[S&P 100]] component|[[S&P 500]] component}} {{New York Stock Exchange|MMM}}',
 'ISIN': '{{ISIN|sl|=|n|pl|=|y|US88579Y1010}}',
 'industry': '[[Conglomerate (company)|Conglomerate]]',
 'foundation': '{{Start date and age|1902|6|13}} in [[Two Harbors, Minnesota]], U.S.',
 'founders': '{{Unbulleted list|J. Danley Budd|Henry S. Bryan|William A. McGonagle|John Dwan|Hermon W. Cable | Charles Simmons|ref|{{cite web |url=https://www.3m.com.au/3M/en_AU/company-au/news-releases/full-story/?storyid=51f5cfac-3ea9-4a98-a406-e2b955c3fd40 |title=It all started with a rock |date=J

## Step 3: Aggregate Infoboxes into Databases

### What We're Building
We're converting individual JSON files (one per company) into consolidated CSV databases (one per index).

### Why?
- **Easier analysis**: CSV format works with pandas, Excel, and most analysis tools
- **Efficiency**: One file per index instead of hundreds of individual JSON files
- **Standardization**: Creates a uniform dataset structure for LLM processing

### Process
1. Read all JSON infobox files for an index from disk
2. Convert each JSON to a DataFrame row
3. Concatenate all rows into a single DataFrame
4. Save as CSV with proper encoding

### Notes for Future Enhancement
- The infoboxes contain many fields beyond what we use now (URLs, images, etc.)
- Future work could extract and leverage additional information
- This foundation allows flexible data extraction later

In [33]:
# ============================================================================
# FUNCTION 4: Extract Maximum Company Infoboxes for Each Index
# ============================================================================
# This function loops through all indices, retrieves the company list from
# Wikipedia or CSV, fetches each company's infobox using wptools, and saves
# JSON files for later aggregation.

def extract_all_index_infoboxes_from_indexes(index_config: dict,
                                             base_json_dir: Path,
                                             delay: float = 0.5,
                                             max_companies: int = 40) -> dict:
    """
    Extract infoboxes for companies from index CSVs defined in INDEX_CONFIG.
    Only process up to `max_companies` companies per index.
    """

    summary = {}

    for index_name, config in index_config.items():
        table_path = config.get("table")
        company_col = config.get("column")

        print(f"\n==============================")
        print(f"Processing index: {index_name.upper()}")
        print("==============================")

        # Directory to save JSON infoboxes
        index_json_dir = base_json_dir / index_name
        index_json_dir.mkdir(parents=True, exist_ok=True)

        # Read the CSV table
        try:
            df = pd.read_csv(table_path)
        except Exception as e:
            print(f"Failed to read table for {index_name}: {e}")
            summary[index_name] = 0
            continue

        if company_col is None or company_col not in df.columns:
            print(f"Column '{company_col}' not found for {index_name}")
            summary[index_name] = 0
            continue

        # Take only the first `max_companies` entries
        companies = list(df[company_col].astype(str).str.strip())[:max_companies]
        companies = list(dict.fromkeys(companies))  # remove duplicates

        # Fetch infoboxes
        success_count = 0
        for company_name_str in tqdm(companies, desc=f"Fetching infoboxes {index_name}", unit="company"):
            json_path = index_json_dir / f"{company_name_str.replace(' ', '_')}.json"
            if json_path.exists():
                continue
            try:
                infobox = fetch_company_infobox(company_name_str, delay=delay)
                if infobox:
                    save_infobox_json(company_name_str, infobox, index_json_dir)
                    success_count += 1
            except Exception:
                continue

        print(f"{success_count} infoboxes saved for {index_name}")
        summary[index_name] = success_count

    return summary


INDEX_CONFIG = {
    "bsesensex":   {"table": "./sample_data/indices/bsesensex/table_2.csv", "column": "Company"},
    "cac40":       {"table": "./sample_data/indices/cac40/table_4.csv", "column": "Company"},
    "csi300":      {"table": "./sample_data/indices/csi300/table_3.csv", "column": "Company"},
    "dax":         {"table": "./sample_data/indices/dax/table_4.csv", "column": "Company"},
    "eurostoxx50": {"table": "./sample_data/indices/eurostoxx50/table_4.csv", "column": "Name"},
    "nasdaq100":   {"table": "./sample_data/indices/nasdaq100/table_4.csv", "column": "Company"},
    "sp500":       {"table": "./sample_data/indices/sp500/table_1.csv", "column": "Security"},
    "spla40":      {"table": "./sample_data/indices/spla40/table_1.csv", "column": "Company name"}
}

base_json_dir = Path("./sample_data/infoboxes")

results = extract_all_index_infoboxes_from_indexes(
    index_config=INDEX_CONFIG,
    base_json_dir=base_json_dir,
    delay=0.5,
    max_companies=40
)

print("\nSummary of extracted infoboxes per index:")
for index_name, count in results.items():
    print(f"{index_name}: {count} infoboxes extracted")


Processing index: BSESENSEX


Fetching infoboxes bsesensex:   0%|          | 0/30 [00:00<?, ?company/s]en.wikipedia.org (parse) Adani Ports & SEZ
Adani Ports & SEZ (en) data
{
  infobox: <dict(25)> name, logo, logo_size, former_name, type, tr...
  pageid: 7468178
  parsetree: <str(68204)> <root><template><title>short description...
  requests: <list(1)> parse
  title: Adani Ports & SEZ
  wikibase: Q16058076
  wikidata_url: https://www.wikidata.org/wiki/Q16058076
  wikitext: <str(50967)> {{short description|Indian multinational ...
}
Fetching infoboxes bsesensex:   3%|▎         | 1/30 [00:01<00:39,  1.35s/company]

✔ Saved infobox: sample_data\infoboxes\bsesensex\Adani_Ports_&_SEZ.json


en.wikipedia.org (parse) Asian Paints
Asian Paints (en) data
{
  infobox: <dict(23)> name, logo, logo_size, former_name, type, tr...
  pageid: 52312262
  parsetree: <str(25090)> <root><template><title>Short description...
  requests: <list(1)> parse
  title: Asian Paints
  wikibase: Q28171825
  wikidata_url: https://www.wikidata.org/wiki/Q28171825
  wikitext: <str(17359)> {{Short description|Indian multinational ...
}
Fetching infoboxes bsesensex:   7%|▋         | 2/30 [00:02<00:29,  1.07s/company]

✔ Saved infobox: sample_data\infoboxes\bsesensex\Asian_Paints.json


...

No infobox found for: Vale


en.wikipedia.org (parse) Wal-Mart de México
en.wikipedia.org (imageinfo) File:Walmart avila camacho .jpg
Walmart de México y Centroamérica (en) data
{
  image: <list(1)> {'kind': 'parse-image', 'file': 'File:Walmart a...
  infobox: <dict(20)> name, logo, logo_size, image, image_caption,...
  iwlinks: <list(2)> https://commons.wikimedia.org/wiki/Category:W...
  pageid: 229705
  parsetree: <str(18372)> <root><template><title>short description...
  requests: <list(2)> parse, imageinfo
  title: Walmart de México y Centroamérica
  wikibase: Q1064887
  wikidata_url: https://www.wikidata.org/wiki/Q1064887
  wikitext: <str(14173)> {{short description|Division of Walmart}}...
}
Fetching infoboxes spla40:  98%|█████████▊| 39/40 [00:43<00:01,  1.05s/company]

✔ Saved infobox: sample_data\infoboxes\spla40\Wal-Mart_de_México.json


en.wikipedia.org (parse) WEG Industries
en.wikipedia.org (imageinfo) File:WEG II, Asa leste.jpg
WEG Industries (en) data
{
  image: <list(1)> {'kind': 'parse-image', 'file': 'File:WEG II, A...
  infobox: <dict(16)> name, logo, logo_size, image, image_caption,...
  pageid: 6467284
  parsetree: <str(12344)> <root><template><title>Short description...
  requests: <list(2)> parse, imageinfo
  title: WEG Industries
  wikibase: Q634794
  wikidata_url: https://www.wikidata.org/wiki/Q634794
  wikitext: <str(8945)> {{Short description|Brazilian electronics ...
}
Fetching infoboxes spla40: 100%|██████████| 40/40 [00:44<00:00,  1.11s/company]

✔ Saved infobox: sample_data\infoboxes\spla40\WEG_Industries.json
32 infoboxes saved for spla40

Summary of extracted infoboxes per index:
bsesensex: 26 infoboxes extracted
cac40: 37 infoboxes extracted
csi300: 37 infoboxes extracted
dax: 36 infoboxes extracted
eurostoxx50: 40 infoboxes extracted
nasdaq100: 37 infoboxes extracted
sp500: 37 infoboxes extracted
spla40: 32 infoboxes extracted


In [34]:
# ============================================================================
# EXECUTION: Merge All Infoboxes into Index Databases
# ============================================================================
# Loop through each index folder and consolidate all JSON infoboxes into CSV

def merge_infoboxes_to_csv(base_json_dir: Path, output_dir: Path) -> None:
    """
    Consolidate all JSON infobox files for each index into a single CSV file.
    
    Parameters
    ----------
    base_json_dir : Path
        Directory containing subfolders for each index with JSON infobox files
    output_dir : Path
        Directory where consolidated CSV files will be saved
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    for index_dir in base_json_dir.iterdir():
        if not index_dir.is_dir():
            continue

        index_name = index_dir.name
        print(f"\nProcessing index: {index_name.upper()}")

        all_data = []

        # Loop through all JSON files in the index folder
        for json_file in index_dir.glob("*.json"):
            try:
                with open(json_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    infobox = data if isinstance(data, dict) else {}
                    if infobox:
                        # Flatten nested dict if needed
                        all_data.append(infobox)
            except Exception as e:
                print(f"Failed to read {json_file}: {e}")
                continue

        if not all_data:
            print(f"No infobox data found for {index_name}")
            continue

        # Convert list of dicts to DataFrame
        df = pd.DataFrame(all_data)

        # Save as CSV
        csv_path = output_dir / f"{index_name}.csv"
        df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Saved consolidated CSV: {csv_path}")

base_json_dir = Path("./sample_data/infoboxes")
output_dir = Path("./sample_data/databases")

merge_infoboxes_to_csv(base_json_dir, output_dir)


Processing index: BSESENSEX
Saved consolidated CSV: sample_data\databases\bsesensex.csv

Processing index: CAC40
Saved consolidated CSV: sample_data\databases\cac40.csv

Processing index: CSI300
Saved consolidated CSV: sample_data\databases\csi300.csv

Processing index: DAX
Saved consolidated CSV: sample_data\databases\dax.csv

Processing index: EUROSTOXX50
Saved consolidated CSV: sample_data\databases\eurostoxx50.csv

Processing index: NASDAQ100
Saved consolidated CSV: sample_data\databases\nasdaq100.csv

Processing index: SP500
Saved consolidated CSV: sample_data\databases\sp500.csv

Processing index: SPLA40
Saved consolidated CSV: sample_data\databases\spla40.csv


# Part 2: Data Processing & LLM Prompt Engineering

## Overview of Part 2
Now that we have structured company data from Wikipedia, we'll:

1. **Load and analyze** the infobox database
2. **Clean and preprocess** the data for LLM consumption
3. **Create prompt templates** for different LLM tasks
4. **Design context formatting** that maximizes LLM effectiveness

## Key Concepts

### Why Clean Data for LLMs?
- LLMs perform better with well-structured, clean text
- Removing noise and formatting artifacts improves accuracy
- Consistent formatting allows for better prompt engineering
- Clean data enables batch processing and cost optimization

### Prompt Engineering
Prompt engineering is the art of crafting inputs to LLMs to get better outputs. We'll explore:
- **Context formatting**: How to present company data effectively
- **Task-specific templates**: Different prompts for different goals
- **Few-shot learning**: Providing examples to guide LLM behavior
- **Output structuring**: Getting structured responses (JSON, tables, etc.)


In [40]:
# ============================================================================
# STEP 4: LOAD AND ANALYZE INFOBOX DATA
# ============================================================================
# Now we'll load the consolidated CSV database and analyze its structure

# Load the S&P 500 infobox data
df_sp500 = pd.read_csv("./sample_data/databases/sp500.csv")

# Display the first few rows
print("First 5 rows of the dataset:")
display(df_sp500.head())

# Dataset information
print("\n=== Dataset Info ===")
df_sp500.info()

# Summary statistics for numerical columns
print("\n=== Numerical Summary ===")
print(df_sp500.describe())

# Missing values per column
print("\n=== Missing Values ===")
missing = df_sp500.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values detected.")

# Analyze categorical columns
categorical_cols = df_sp500.select_dtypes(include='object').columns.tolist()
if categorical_cols:
    print("\n=== Categorical Column Overview ===")
    for col in categorical_cols:
        unique_vals = df_sp500[col].dropna().unique()
        n_unique = len(unique_vals)
        preview = ', '.join(map(str, unique_vals[:10]))
        print(f"- {col} ({n_unique} unique values): {preview}{'...' if n_unique > 10 else ''}")
else:
    print("\nNo categorical columns found.")


First 5 rows of the dataset:


,name,logo,logo_size,image,image_size,image_caption,former_name,type,traded_as,ISIN,...,predecessors,owner,module,homepage,parent,aum,former_names,logo_upright,production,num_locations_year
0,3M Company,3M wordmark.svg,175px,3-M Building Maplewood MN1.jpg,250px,"3M headquarters in [[Maplewood, Minnesota]]",Minnesota Mining and Manufacturing Company (19...,[[Public company|Public]],{{Unbulleted list|New York Stock Exchange|MMM|...,{{ISIN|sl|=|n|pl|=|y|US88579Y1010}},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A. O. Smith Corporation,AO Smith logo.svg {{!}} class=skin-invert-image,NaN,A. O. Smith Corporate Technology Center.jpg,NaN,"A. O. Smith Corporate Technology Center, opene...",NaN,[[Public company|Public]],{{Unbulleted list|NYSE|AOS|[[S&P 500]] compone...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Abbott Laboratories,Abbott Laboratories 2025 logo.svg,NaN,NaN,NaN,NaN,NaN,[[Public company|Public]],{{ubl|NYSE|ABT|[[S&P 100]] component|[[S&P 500...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AbbVie Inc.,[[File:AbbVie logo.svg|frameless|upright=0.9|c...,NaN,NaN,NaN,NaN,NaN,[[Public company|Public]],{{ubl|NYSE|ABBV|[[S&P 100]] component|[[S&P 50...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Accenture plc,Accenture.svg,NaN,Grand Canal Square - panoramio.jpg,250px,Headquarters at 1 [[Grand Canal Dock#Grand Can...,Andersen Consulting,[[Public limited company|Public]],{{ubl|NYSE|ACN| ([[Class A share|Class A]])|[[...,{{ISIN|sl|=|n|pl|=|y|IE00B4BNMY34}},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 63 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   name                 37 non-null     object 
 1   logo                 36 non-null     object 
 2   logo_size            10 non-null     object 
 3   image                21 non-null     object 
 4   image_size           9 non-null      object 
 5   image_caption        21 non-null     object 
 6   former_name          7 non-null      object 
 7   type                 37 non-null     object 
 8   traded_as            37 non-null     object 
 9   ISIN                 8 non-null      object 
 10  industry             37 non-null     object 
 11  foundation           8 non-null      object 
 12  founders             13 non-null     object 
 13  location_city        2 non-null      object 
 14  location_country     2 non-null      object 
 15  area_served         

In [43]:
# --------------------------
# 1. Clean text function
# --------------------------
def clean_text(text: str) -> str:
    """
    Clean and normalize text for LLM input.

    Steps:
    - Replace newlines/tabs with a single space
    - Collapse multiple spaces
    - Remove redundant punctuation (more than 2 consecutive)
    - Strip leading/trailing spaces
    """
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r"[\n\t]+", " ", text)           # Replace newlines/tabs
    text = re.sub(r"\s+", " ", text)              # Collapse multiple spaces
    text = re.sub(r"([,.!?]){2,}", r"\1", text)   # Reduce repeated punctuation
    
    return text.strip()

# --------------------------
# 2. Extract key facts
# --------------------------
def extract_key_facts(row: pd.Series) -> Dict[str, str]:
    """
    Extract key facts from a company row (infobox).

    Uses case-insensitive matching for field names. Returns
    a dictionary with empty string for missing fields.
    """
    key_fields = {
        "name": ["name", "company", "security", "company name"],
        "type": ["type", "company type"],
        "industry": ["industry", "sector"],
        "founded": ["founded", "established"],
        "founder": ["founder", "founders"],
        "headquarters": ["headquarters", "hq", "location"],
        "key_people": ["key_people", "people", "leadership"],
        "employees": ["employees", "staff"],
        "revenue": ["revenue", "income", "turnover"],
        "website": ["website", "url"],
        "stock_exchange": ["stock_exchange", "exchange", "symbol"]
    }
    
    facts = {}
    columns_lower = {col.lower(): col for col in row.index}
    
    for key, candidates in key_fields.items():
        for candidate in candidates:
            col_match = columns_lower.get(candidate.lower())
            if col_match:
                value = row[col_match]
                facts[key] = clean_text(str(value)) if pd.notna(value) else ""
                break
        else:
            facts[key] = ""
    
    return facts

# --------------------------
# 3. Convert row to LLM context
# --------------------------
def row_to_context(row: pd.Series, include_empty: bool = False) -> str:
    """
    Convert a company row into a formatted text block for LLM input.

    Parameters
    ----------
    row : pd.Series
        A single company row.
    include_empty : bool
        Whether to include fields with empty values.

    Returns
    -------
    str
        Formatted multi-line context string.
    """
    facts = extract_key_facts(row)
    context_lines = ["COMPANY INFORMATION:", "--"]
    
    # Optional: define a consistent order of fields
    ordered_fields = [
        "name", "type", "industry", "founded", "founder",
        "headquarters", "key_people", "employees", "revenue",
        "website", "stock_exchange"
    ]
    
    for field in ordered_fields:
        value = facts.get(field, "")
        if value or include_empty:
            context_lines.append(f"{field.upper()}: {value}")
    
    return "\n".join(context_lines)

# ============================================================================ 
# TEST: Demonstrate preprocessing on sample company
# ============================================================================

# Take first row of S&P 500 dataset
sample_row = df_sp500.iloc[0]
context_text = row_to_context(sample_row)

print("Formatted context for LLM:\n")
print(context_text)

Formatted context for LLM:

COMPANY INFORMATION:
--
NAME: 3M Company
TYPE: [[Public company|Public]]
INDUSTRY: [[Conglomerate (company)|Conglomerate]]
KEY_PEOPLE: {{plainlist| * [[Michael F. Roman]] (chairman) * [[William M. Brown (businessman)|William M. Brown]] (CEO)|ref|{{citation|url=https://www.manufacturingdive.com/news/3m-ceo-william-brown-executive-chair-michael-roman/709987/|website=Manufacturing Dive|date=March 12, 2024 |title=3M appoints new CEO }}|</ref>}}
REVENUE: {{decrease}} {{US$|24.58 billion|link|=|yes}} (2024)
WEBSITE: {{URL|3m.com}}


Create a PromptBuilder class with different methods for creating prompts:
- qa prompt
- classification prompt
- summarization prompt
- comparison prompt
- information extraction prompt

In [52]:
# ============================================================================
# STEP 6: CREATE PROMPT TEMPLATES FOR LLM TASKS
# ============================================================================

class PromptBuilder:
    """
    Enhanced PromptBuilder for LLM tasks following best practices.
    
    Features:
    - Metadata inclusion in context
    - Few-shot examples for classification and extraction
    - Explicit output formats
    - Role/perspective for better results
    """

    @staticmethod
    def add_metadata(context: str, source: str = "CSV dataset", date: str = None) -> str:
        """Add metadata to context."""
        date_str = date or datetime.today().strftime("%Y-%m-%d")
        return f"{context}\n\nSOURCE: {source}\nDATE: {date_str}"

    @staticmethod
    def qa_prompt(context: str, question: str) -> str:
        """Build a QA prompt with metadata and role."""
        context = PromptBuilder.add_metadata(context)
        prompt = (
            "You are a knowledgeable financial analyst.\n"
            "Use the company information below to answer the question accurately.\n\n"
            f"{context}\n\n"
            f"QUESTION: {question}\n"
            "ANSWER:"
        )
        return prompt

    @staticmethod
    def classification_prompt(context: str, categories: list, few_shot_examples: list = None) -> str:
        """Build a classification prompt with optional few-shot examples and JSON output."""
        context = PromptBuilder.add_metadata(context)
        categories_str = ", ".join(categories)
        examples_text = ""
        if few_shot_examples:
            examples_text = "Here are some examples:\n"
            for ex_context, ex_category in few_shot_examples:
                examples_text += f"{ex_context}\nCATEGORY: {ex_category}\n\n"

        prompt = (
            "You are a financial classification expert.\n"
            "Classify the company into one of the following categories (output as JSON):\n"
            f"{categories_str}\n\n"
            f"{examples_text}"
            f"{context}\n\n"
            "CATEGORY (JSON format):"
        )
        return prompt

    @staticmethod
    def summarization_prompt(context: str, max_words: int = 200) -> str:
        """Build a summarization prompt with metadata and word limit."""
        context = PromptBuilder.add_metadata(context)
        prompt = (
            "Summarize the key information about the company below.\n"
            f"Keep the summary concise, under {max_words} words.\n\n"
            f"{context}\n\n"
            "SUMMARY:"
        )
        return prompt

    @staticmethod
    def comparison_prompt(context1: str, context2: str, aspects: list = None) -> str:
        """Build a comparison prompt with optional aspects and metadata."""
        context1 = PromptBuilder.add_metadata(context1)
        context2 = PromptBuilder.add_metadata(context2)
        aspects_str = f" Compare based on: {', '.join(aspects)}." if aspects else ""
        prompt = (
            "You are a financial analyst.\n"
            f"Compare the following two companies.{aspects_str}\n\n"
            f"COMPANY 1:\n{context1}\n\n"
            f"COMPANY 2:\n{context2}\n\n"
            "COMPARISON (structured, concise):"
        )
        return prompt

    @staticmethod
    def extraction_prompt(context: str, fields: list, few_shot_examples: list = None) -> str:
        """Build an information extraction prompt with few-shot examples and JSON output."""
        context = PromptBuilder.add_metadata(context)
        fields_str = ", ".join(fields)
        examples_text = ""
        if few_shot_examples:
            examples_text = "Here are some examples:\n"
            for ex_context, ex_output in few_shot_examples:
                examples_text += f"{ex_context}\nEXTRACTED INFORMATION: {ex_output}\n\n"

        prompt = (
            "Extract the following information from the company data below (output as JSON):\n"
            f"{fields_str}\n\n"
            f"{examples_text}"
            f"{context}\n\n"
            "EXTRACTED INFORMATION (JSON format):"
        )
        return prompt

In [54]:
# ============================================================================
# TEST: Demonstrate Prompt Creation for a Sample Company
# ============================================================================

builder = PromptBuilder()

# Take the first row from S&P 500 dataset and convert to context
sample_context = row_to_context(df_sp500.iloc[0])

# --------------------------
# 1. Question-Answering Prompt
# --------------------------
print("\n" + "="*20 + " QA PROMPT " + "="*20 + "\n")
qa_question = "What industry does this company operate in?"
qa_prompt = builder.qa_prompt(sample_context, qa_question)
print(qa_prompt)

# --------------------------
# 2. Classification Prompt
# --------------------------
print("\n" + "="*20 + " CLASSIFICATION PROMPT " + "="*20 + "\n")
categories = ["Technology", "Finance", "Healthcare", "Industrial"]
classification_prompt = builder.classification_prompt(sample_context, categories)
print(classification_prompt)

# --------------------------
# 3. Summarization Prompt
# --------------------------
print("\n" + "="*20 + " SUMMARIZATION PROMPT " + "="*20 + "\n")
summarization_prompt = builder.summarization_prompt(sample_context, max_words=150)
print(summarization_prompt)

# --------------------------
# 4. Information Extraction Prompt
# --------------------------
print("\n" + "="*20 + " EXTRACTION PROMPT " + "="*20 + "\n")
fields_to_extract = ["founder", "headquarters", "revenue"]
extraction_prompt = builder.extraction_prompt(sample_context, fields_to_extract)
print(extraction_prompt)


==================== QA PROMPT ====================

You are a knowledgeable financial analyst.
Use the company information below to answer the question accurately.

COMPANY INFORMATION:
--
NAME: 3M Company
TYPE: [[Public company|Public]]
INDUSTRY: [[Conglomerate (company)|Conglomerate]]
KEY_PEOPLE: {{plainlist| * [[Michael F. Roman]] (chairman) * [[William M. Brown (businessman)|William M. Brown]] (CEO)|ref|{{citation|url=https://www.manufacturingdive.com/news/3m-ceo-william-brown-executive-chair-michael-roman/709987/|website=Manufacturing Dive|date=March 12, 2024 |title=3M appoints new CEO }}|</ref>}}
REVENUE: {{decrease}} {{US$|24.58 billion|link|=|yes}} (2024)
WEBSITE: {{URL|3m.com}}

SOURCE: CSV dataset
DATE: 2025-11-28

QUESTION: What industry does this company operate in?
ANSWER:

==================== CLASSIFICATION PROMPT ====================

You are a financial classification expert.
Classify the company into one of the following categories (output as JSON):
Technology, Fina

ChatGPT's response to the prompt test:

==================== QA PROMPT ====================

ANSWER: 3M Company operates in the conglomerate industry, which spans multiple sectors including industrial, safety, consumer goods, and healthcare products.

==================== CLASSIFICATION PROMPT ====================

{
  "category": "Industrial"
}

==================== SUMMARIZATION PROMPT ====================

SUMMARY: 3M Company is a publicly traded conglomerate generating $24.58 billion in revenue in 2024. Michael F. Roman serves as chairman, and William M. Brown is the CEO. The company operates across multiple sectors, including industrial, healthcare, and consumer products. More information is available on its website: [3m.com](https://www.3m.com/).

==================== EXTRACTION PROMPT ====================

{
  "founder": null,
  "headquarters": null,
  "revenue": "US$24.58 billion (2024)"
}

In [39]:
# ============================================================================
# STEP 7: SUMMARY & BEST PRACTICES GUIDE
# ============================================================================
# Comprehensive guide to using Wikipedia data with LLMs

summary = """
╔════════════════════════════════════════════════════════════════════════════╗
║                    LLM PROMPT ENGINEERING SUMMARY                         ║
╚════════════════════════════════════════════════════════════════════════════╝


🎯 BEST PRACTICES FOR LLM PROMPT INJECTION:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. CONTEXT QUALITY
   • Keep context focused and relevant
   • Clean and normalize text thoroughly
   • Remove ambiguous or conflicting information
   • Include metadata (source, confidence, date)

2. PROMPT DESIGN
   • Use clear, specific instructions
   • Provide examples when possible (few-shot)
   • Specify output format explicitly (JSON, tables, etc.)
   • Include role/perspective for better results



📋 USE CASES FOR YOUR DATA:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✓ Company Classification        → Categorize by industry, size, sector
✓ Market Analysis               → Competitive landscape, positioning
✓ Risk Assessment               → Financial health, strategic risks
✓ Investment Analysis           → Potential returns, growth prospects
✓ Data Enrichment               → Fill gaps from Wikipedia data
✓ Text Generation               → Create summaries, reports, profiles
✓ Knowledge Extraction          → Key metrics, relationships, entities
✓ Sentiment Analysis            → Company reputation, public perception
✓ Trend Detection               → Emerging patterns, growth areas
✓ Comparative Analysis          → Company benchmarking, peer analysis
"""

End of lab 5

🚀 NEXT STEPS (in anticipation of the final lab)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Export your full dataset using PromptExporter
2. Test prompts with a small sample (5-10 companies)
3. Evaluate LLM outputs for quality and accuracy
4. Iterate on prompts based on results
5. Scale up to full dataset using batch APIs
6. Monitor token usage and costs
7. Implement feedback loops for continuous improvement
8. Build evaluation metrics for output quality